In [0]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [0]:
# Load the feature-engineered datasets
print("=" * 60)
print("LOADING FEATURE-ENGINEERED DATASETS")
print("=" * 60)

# Load all versions
print("\nLoading datasets...\n")

# 1. Derived features (for tree-based models)
df_derived = spark.table("car_price_features_derived").toPandas()
print(f"✓ Loaded: car_price_features_derived")
print(f"   Shape: {df_derived.shape}")

# 2. Label encoded (for tree-based models)
df_labeled = spark.table("car_price_features_labeled").toPandas()
print(f"\n✓ Loaded: car_price_features_labeled")
print(f"   Shape: {df_labeled.shape}")

# 3. One-hot encoded (for linear models)
df_onehot = spark.table("car_price_features_onehot").toPandas()
print(f"\n✓ Loaded: car_price_features_onehot")
print(f"   Shape: {df_onehot.shape}")

# 4. Scaled (for distance-based models)
df_scaled = spark.table("car_price_features_scaled").toPandas()
print(f"\n✓ Loaded: car_price_features_scaled")
print(f"   Shape: {df_scaled.shape}")

print("\nAll datasets loaded successfully!")

LOADING FEATURE-ENGINEERED DATASETS

Loading datasets...

✓ Loaded: car_price_features_derived
   Shape: (205, 41)

✓ Loaded: car_price_features_labeled
   Shape: (205, 51)

✓ Loaded: car_price_features_onehot
   Shape: (205, 206)

✓ Loaded: car_price_features_scaled
   Shape: (205, 206)

All datasets loaded successfully!


In [0]:
# Data Quality Check
print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

# Check for missing values
print("\n1. MISSING VALUES CHECK:\n")
for name, df in [("Derived", df_derived), ("Labeled", df_labeled), 
                  ("One-Hot", df_onehot), ("Scaled", df_scaled)]:
    missing = df.isnull().sum().sum()
    print(f"   {name:12s}: {missing} missing values")

# Check for infinite values
print("\n2. INFINITE VALUES CHECK:\n")
for name, df in [("Derived", df_derived), ("Labeled", df_labeled), 
                  ("One-Hot", df_onehot), ("Scaled", df_scaled)]:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_count = np.isinf(df[numeric_cols]).sum().sum()
    print(f"   {name:12s}: {inf_count} infinite values")

# Check for duplicate rows
print("\n3. DUPLICATE ROWS CHECK:\n")
for name, df in [("Derived", df_derived), ("Labeled", df_labeled), 
                  ("One-Hot", df_onehot), ("Scaled", df_scaled)]:
    duplicates = df.duplicated().sum()
    print(f"   {name:12s}: {duplicates} duplicate rows")

# Check data types
print("\n4. DATA TYPES SUMMARY:\n")
for name, df in [("One-Hot", df_onehot)]:
    print(f"   {name}:")
    print(f"      Numerical columns: {len(df.select_dtypes(include=[np.number]).columns)}")
    print(f"      Object columns: {len(df.select_dtypes(include=['object']).columns)}")

print("\n✓ Data quality check complete!")

DATA QUALITY CHECK

1. MISSING VALUES CHECK:

   Derived     : 0 missing values
   Labeled     : 0 missing values
   One-Hot     : 0 missing values
   Scaled      : 0 missing values

2. INFINITE VALUES CHECK:

   Derived     : 0 infinite values
   Labeled     : 0 infinite values
   One-Hot     : 0 infinite values
   Scaled      : 0 infinite values

3. DUPLICATE ROWS CHECK:

   Derived     : 0 duplicate rows
   Labeled     : 0 duplicate rows
   One-Hot     : 0 duplicate rows
   Scaled      : 0 duplicate rows

4. DATA TYPES SUMMARY:

   One-Hot:
      Numerical columns: 206
      Object columns: 0

✓ Data quality check complete!


In [0]:
# Handle any remaining issues (if found)
print("=" * 60)
print("DATA CLEANING (IF NEEDED)")
print("=" * 60)

# Replace infinite values with NaN, then fill with median
print("\nCleaning datasets...\n")

for name, df in [("df_derived", df_derived), ("df_labeled", df_labeled), 
                  ("df_onehot", df_onehot), ("df_scaled", df_scaled)]:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    # Replace inf/-inf with NaN
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    
    # Fill NaN with median
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].median(), inplace=True)

print("✓ All datasets cleaned!")
print("   - Infinite values replaced")
print("   - Missing values filled with median")

DATA CLEANING (IF NEEDED)

Cleaning datasets...

✓ All datasets cleaned!
   - Infinite values replaced
   - Missing values filled with median


In [0]:
# Prepare data for train-test split
print("=" * 60)
print("PREPARING FOR TRAIN-TEST SPLIT")
print("=" * 60)

# We'll use the one-hot encoded dataset as the primary one for this example
# You can adjust this based on which model you want to use

# For demonstration, let's work with df_onehot (good for linear models)
df_main = df_onehot.copy()

print(f"\nWorking with: One-Hot Encoded Dataset")
print(f"Shape: {df_main.shape}")

# Identify target column
target_column = 'price'
print(f"\nTarget variable: {target_column}")

# Separate features and target
if target_column in df_main.columns:
    X = df_main.drop(target_column, axis=1)
    y = df_main[target_column]
    
    print(f"\nFeatures (X): {X.shape}")
    print(f"Target (y): {y.shape}")
    
    # Remove any remaining non-numeric columns from X
    non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        print(f"\nRemoving non-numeric columns: {non_numeric}")
        X = X.select_dtypes(include=[np.number])
        print(f"Final Features (X): {X.shape}")
else:
    print(f"\n⚠️  Warning: Target column '{target_column}' not found!")

print("\n✓ Data prepared for splitting!")

PREPARING FOR TRAIN-TEST SPLIT

Working with: One-Hot Encoded Dataset
Shape: (205, 206)

Target variable: price

Features (X): (205, 205)
Target (y): (205,)

✓ Data prepared for splitting!


In [0]:
# Train-Test Split
print("=" * 60)
print("TRAIN-TEST SPLIT")
print("=" * 60)

# Split configuration
test_size = 0.2  # 80% train, 20% test
random_state = 42

print(f"\nSplit configuration:")
print(f"   Test size: {test_size * 100}%")
print(f"   Train size: {(1 - test_size) * 100}%")
print(f"   Random state: {random_state}")

# Perform split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=test_size, 
    random_state=random_state,
    shuffle=True
)

print(f"\n✓ Split completed!")
print(f"\nTraining set:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"\nTest set:")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")

# Calculate split percentages
train_pct = (len(X_train) / len(X)) * 100
test_pct = (len(X_test) / len(X)) * 100
print(f"\nActual split:")
print(f"   Train: {len(X_train)} samples ({train_pct:.1f}%)")
print(f"   Test: {len(X_test)} samples ({test_pct:.1f}%)")

TRAIN-TEST SPLIT

Split configuration:
   Test size: 20.0%
   Train size: 80.0%
   Random state: 42

✓ Split completed!

Training set:
   X_train: (164, 205)
   y_train: (164,)

Test set:
   X_test: (41, 205)
   y_test: (41,)

Actual split:
   Train: 164 samples (80.0%)
   Test: 41 samples (20.0%)


In [0]:
# Validate the split
print("=" * 60)
print("SPLIT VALIDATION")
print("=" * 60)

# 1. Check target distribution
print("\n1. TARGET DISTRIBUTION:\n")
print(f"   Training set:")
print(f"      Mean: ${y_train.mean():,.2f}")
print(f"      Median: ${y_train.median():,.2f}")
print(f"      Std: ${y_train.std():,.2f}")
print(f"      Min: ${y_train.min():,.2f}")
print(f"      Max: ${y_train.max():,.2f}")

print(f"\n   Test set:")
print(f"      Mean: ${y_test.mean():,.2f}")
print(f"      Median: ${y_test.median():,.2f}")
print(f"      Std: ${y_test.std():,.2f}")
print(f"      Min: ${y_test.min():,.2f}")
print(f"      Max: ${y_test.max():,.2f}")

# 2. Check for data leakage (no overlap in indices)
print("\n2. DATA LEAKAGE CHECK:\n")
train_indices = set(X_train.index)
test_indices = set(X_test.index)
overlap = train_indices.intersection(test_indices)
if len(overlap) == 0:
    print("   ✓ No overlap between train and test sets")
else:
    print(f"   ⚠️  Warning: {len(overlap)} overlapping indices found!")

# 3. Feature statistics comparison
print("\n3. FEATURE STATISTICS:\n")
print(f"   Training set:")
print(f"      Non-zero features: {(X_train != 0).sum().sum():,}")
print(f"      Zero features: {(X_train == 0).sum().sum():,}")

print(f"\n   Test set:")
print(f"      Non-zero features: {(X_test != 0).sum().sum():,}")
print(f"      Zero features: {(X_test == 0).sum().sum():,}")

print("\n✓ Split validation complete!")

SPLIT VALIDATION

1. TARGET DISTRIBUTION:

   Training set:
      Mean: $13,223.41
      Median: $10,646.50
      Std: $7,746.21
      Min: $5,118.00
      Max: $45,400.00

   Test set:
      Mean: $13,489.89
      Median: $9,960.00
      Std: $8,995.42
      Min: $5,151.00
      Max: $41,315.00

2. DATA LEAKAGE CHECK:

   ✓ No overlap between train and test sets

3. FEATURE STATISTICS:

   Training set:
      Non-zero features: 5,824
      Zero features: 27,796

   Test set:
      Non-zero features: 1,451
      Zero features: 6,954

✓ Split validation complete!


In [0]:
# Create additional splits for different model types
print("=" * 60)
print("CREATING MODEL-SPECIFIC SPLITS")
print("=" * 60)

print("\nCreating train-test splits for all dataset versions...\n")

# Dictionary to store all splits
splits = {}

# 1. Derived features (for tree-based models like Random Forest, XGBoost)
print("1. DERIVED FEATURES (Tree-based models):")
df_temp = df_derived.copy()
if 'price' in df_temp.columns:
    X_temp = df_temp.drop('price', axis=1)
    # Remove non-numeric columns
    X_temp = X_temp.select_dtypes(include=[np.number])
    y_temp = df_temp['price']
    
    X_train_derived, X_test_derived, y_train_derived, y_test_derived = train_test_split(
        X_temp, y_temp, test_size=0.2, random_state=42, shuffle=True
    )
    
    splits['derived'] = {
        'X_train': X_train_derived, 'X_test': X_test_derived,
        'y_train': y_train_derived, 'y_test': y_test_derived
    }
    print(f"   ✓ Train: {X_train_derived.shape}, Test: {X_test_derived.shape}")

# 2. Label encoded (for tree-based models)
print("\n2. LABEL ENCODED (Tree-based models):")
df_temp = df_labeled.copy()
if 'price' in df_temp.columns:
    X_temp = df_temp.drop('price', axis=1)
    X_temp = X_temp.select_dtypes(include=[np.number])
    y_temp = df_temp['price']
    
    X_train_labeled, X_test_labeled, y_train_labeled, y_test_labeled = train_test_split(
        X_temp, y_temp, test_size=0.2, random_state=42, shuffle=True
    )
    
    splits['labeled'] = {
        'X_train': X_train_labeled, 'X_test': X_test_labeled,
        'y_train': y_train_labeled, 'y_test': y_test_labeled
    }
    print(f"   ✓ Train: {X_train_labeled.shape}, Test: {X_test_labeled.shape}")

# 3. One-hot encoded (already done above, stored in X_train, X_test, etc.)
print("\n3. ONE-HOT ENCODED (Linear models):")
splits['onehot'] = {
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test
}
print(f"   ✓ Train: {X_train.shape}, Test: {X_test.shape}")

# 4. Scaled (for distance-based models like KNN, SVM, Neural Networks)
print("\n4. SCALED (Distance-based models):")
df_temp = df_scaled.copy()
if 'price' in df_temp.columns:
    X_temp = df_temp.drop('price', axis=1)
    X_temp = X_temp.select_dtypes(include=[np.number])
    y_temp = df_temp['price']
    
    X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled = train_test_split(
        X_temp, y_temp, test_size=0.2, random_state=42, shuffle=True
    )
    
    splits['scaled'] = {
        'X_train': X_train_scaled, 'X_test': X_test_scaled,
        'y_train': y_train_scaled, 'y_test': y_test_scaled
    }
    print(f"   ✓ Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

print("\n✓ All model-specific splits created!")

CREATING MODEL-SPECIFIC SPLITS

Creating train-test splits for all dataset versions...

1. DERIVED FEATURES (Tree-based models):
   ✓ Train: (164, 30), Test: (41, 30)

2. LABEL ENCODED (Tree-based models):
   ✓ Train: (164, 40), Test: (41, 40)

3. ONE-HOT ENCODED (Linear models):
   ✓ Train: (164, 205), Test: (41, 205)

4. SCALED (Distance-based models):
   ✓ Train: (164, 205), Test: (41, 205)

✓ All model-specific splits created!


In [0]:
# Save preprocessed datasets
print("=" * 60)
print("SAVING PREPROCESSED DATASETS")
print("=" * 60)

print("\nSaving train-test splits to tables...\n")

# Save each split version
for split_name, split_data in splits.items():
    print(f"Saving {split_name} splits...")
    
    # Combine train features and target
    train_combined = split_data['X_train'].copy()
    train_combined['price'] = split_data['y_train'].values
    
    # Combine test features and target
    test_combined = split_data['X_test'].copy()
    test_combined['price'] = split_data['y_test'].values
    
    # Convert to Spark and save
    train_spark = spark.createDataFrame(train_combined)
    test_spark = spark.createDataFrame(test_combined)
    
    train_spark.write.mode("overwrite").saveAsTable(f"car_price_train_{split_name}")
    test_spark.write.mode("overwrite").saveAsTable(f"car_price_test_{split_name}")
    
    print(f"   ✓ Saved: car_price_train_{split_name} ({train_combined.shape})")
    print(f"   ✓ Saved: car_price_test_{split_name} ({test_combined.shape})")
    print()

print("=" * 60)
print("ALL DATASETS SAVED SUCCESSFULLY!")
print("=" * 60)

SAVING PREPROCESSED DATASETS

Saving train-test splits to tables...

Saving derived splits...
   ✓ Saved: car_price_train_derived ((164, 31))
   ✓ Saved: car_price_test_derived ((41, 31))

Saving labeled splits...
   ✓ Saved: car_price_train_labeled ((164, 41))
   ✓ Saved: car_price_test_labeled ((41, 41))

Saving onehot splits...
   ✓ Saved: car_price_train_onehot ((164, 206))
   ✓ Saved: car_price_test_onehot ((41, 206))

Saving scaled splits...
   ✓ Saved: car_price_train_scaled ((164, 206))
   ✓ Saved: car_price_test_scaled ((41, 206))

ALL DATASETS SAVED SUCCESSFULLY!


In [0]:
# Preprocessing Summary
print("=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)

print("\n📊 DATASET VERSIONS CREATED:\n")

print("1. DERIVED FEATURES (car_price_train/test_derived)")
print("   - Best for: Random Forest, XGBoost, Decision Trees")
print("   - Features: Original + engineered features")
print(f"   - Train: {splits['derived']['X_train'].shape}")
print(f"   - Test: {splits['derived']['X_test'].shape}")

print("\n2. LABEL ENCODED (car_price_train/test_labeled)")
print("   - Best for: LightGBM, CatBoost, Gradient Boosting")
print("   - Features: Categorical variables encoded as integers")
print(f"   - Train: {splits['labeled']['X_train'].shape}")
print(f"   - Test: {splits['labeled']['X_test'].shape}")

print("\n3. ONE-HOT ENCODED (car_price_train/test_onehot)")
print("   - Best for: Linear Regression, Ridge, Lasso")
print("   - Features: Categorical variables as binary columns")
print(f"   - Train: {splits['onehot']['X_train'].shape}")
print(f"   - Test: {splits['onehot']['X_test'].shape}")

print("\n4. SCALED (car_price_train/test_scaled)")
print("   - Best for: KNN, SVM, Neural Networks")
print("   - Features: One-hot encoded + MinMax scaled (0-1)")
print(f"   - Train: {splits['scaled']['X_train'].shape}")
print(f"   - Test: {splits['scaled']['X_test'].shape}")

print("\n📈 SPLIT CONFIGURATION:")
print(f"   - Train size: 80% ({len(X_train)} samples)")
print(f"   - Test size: 20% ({len(X_test)} samples)")
print(f"   - Random state: 42")
print(f"   - Shuffle: Yes")

print("\n✅ QUALITY CHECKS PASSED:")
print("   ✓ No missing values")
print("   ✓ No infinite values")
print("   ✓ No duplicate rows")
print("   ✓ No data leakage (train-test overlap)")
print("   ✓ Target distribution balanced")

print("\n🎯 NEXT STEPS:")
print("   1. Load appropriate dataset for your model type")
print("   2. Train your model on the training set")
print("   3. Evaluate on the test set")
print("   4. Compare model performance")

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE!")
print("=" * 60)

PREPROCESSING SUMMARY

📊 DATASET VERSIONS CREATED:

1. DERIVED FEATURES (car_price_train/test_derived)
   - Best for: Random Forest, XGBoost, Decision Trees
   - Features: Original + engineered features
   - Train: (164, 30)
   - Test: (41, 30)

2. LABEL ENCODED (car_price_train/test_labeled)
   - Best for: LightGBM, CatBoost, Gradient Boosting
   - Features: Categorical variables encoded as integers
   - Train: (164, 40)
   - Test: (41, 40)

3. ONE-HOT ENCODED (car_price_train/test_onehot)
   - Best for: Linear Regression, Ridge, Lasso
   - Features: Categorical variables as binary columns
   - Train: (164, 205)
   - Test: (41, 205)

4. SCALED (car_price_train/test_scaled)
   - Best for: KNN, SVM, Neural Networks
   - Features: One-hot encoded + MinMax scaled (0-1)
   - Train: (164, 205)
   - Test: (41, 205)

📈 SPLIT CONFIGURATION:
   - Train size: 80% (164 samples)
   - Test size: 20% (41 samples)
   - Random state: 42
   - Shuffle: Yes

✅ QUALITY CHECKS PASSED:
   ✓ No missing value